# 05 — Pre-registered follow-up: centred boundary-aware decoding

See `PREREG_followup.md` (hash frozen in `results/followup_protocol.json` before this ran). Switch cost becomes `max(0, λ − γ·(log b_t − log mean b))`, so a likely boundary *lowers* the cost; the λ grid is widened to {1/16 … 16}. Same cached SetSeq predictions, same nested protocol. Motivated by exp_1's outer results, so not blind.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp1_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_1')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
import exp1_common as C
import exp1_followup as F

In [2]:
protocol = F.verify_prereg()
print('pre-registration verified', protocol['prereg_sha256'][:16], protocol['arms'])

pre-registration verified 1d9211c2c00f6d90 {'setseq_const_wide': [0.0], 'setseq_cbd_wide': [0.0, 0.5, 1.0, 2.0]}


In [3]:
for variant in ('setseq_const_wide', 'setseq_cbd_wide'):
    t0 = time.time()
    result = F.evaluate(variant)
    C.write_json(C.RESULTS / f'variant_{variant}.json', result)
    frame = pd.DataFrame(result['rows'])
    print(f'== {variant}  mean spellF1 {frame.spell_f1.mean():.4f}  ({time.time() - t0:.0f}s)')
    print(frame[['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'gamma', 'lambda_team', 'lambda_dead']].round(4).to_string(index=False))
    print('final config:', result['final_config'])

== setseq_const_wide  mean spellF1 0.6556  (41s)
 match  spell_f1  macro_f1  n_true  n_pred  gamma  lambda_team  lambda_dead
J03WMX    0.6168    0.9196     511     336    0.0          1.0         16.0
J03WN1    0.6439    0.9042     500     356    0.0          1.0          8.0
J03WOH    0.6750    0.9203     402     318    0.0          0.5          1.0
J03WOY    0.6657    0.9199     456     372    0.0          1.0          2.0
J03WPY    0.6832    0.9273     421     343    0.0          1.0          4.0
J03WQQ    0.6883    0.9211     401     383    0.0          1.0          8.0
J03WR9    0.6162    0.9124     370     285    0.0          1.0          2.0
final config: {'temperature': 1.3028667333854622, 'lambda_team': 0.5, 'lambda_dead': 0.125, 'gamma': 0.0}


== setseq_cbd_wide  mean spellF1 0.6569  (151s)
 match  spell_f1  macro_f1  n_true  n_pred  gamma  lambda_team  lambda_dead
J03WMX    0.6168    0.9196     511     336    0.0          1.0         16.0
J03WN1    0.6439    0.9042     500     356    0.0          1.0          8.0
J03WOH    0.6750    0.9203     402     318    0.0          0.5          1.0
J03WOY    0.6613    0.9197     456     369    0.5          2.0          2.0
J03WPY    0.6856    0.9270     421     341    0.5          2.0          4.0
J03WQQ    0.6950    0.9220     401     376    0.5          2.0          8.0
J03WR9    0.6205    0.9124     370     285    0.5          2.0          2.0
final config: {'temperature': 1.3028667333854622, 'lambda_team': 0.5, 'lambda_dead': 0.125, 'gamma': 0.0}


## Pre-registered decisions

In [4]:
rows = pd.concat([pd.read_csv(C.RESULTS / 'rule_rows.csv')] + [
    pd.DataFrame(C.read_json(C.RESULTS / f'variant_{v}.json')['rows'])
    for v in ('setseq_const', 'setseq_bd', 'setseq_const_wide', 'setseq_cbd_wide')], ignore_index=True)
decision = F.decide(rows)
C.write_json(C.RESULTS / 'followup_decision.json', decision)
print(json.dumps(decision, indent=1))

{
 "rules": {
  "rule1_centring_helps": {
   "a": "setseq_cbd_wide",
   "b": "setseq_const_wide",
   "mean_delta": 0.001277325631944318,
   "ci95": [
    -0.0009297727073634647,
    0.003819886699793797
   ],
   "wins": 3,
   "decision": "inconclusive"
  },
  "rule2_grid_edge_mattered": {
   "a": "setseq_const_wide",
   "b": "setseq_const",
   "mean_delta": 0.0,
   "ci95": [
    0.0,
    0.0
   ],
   "wins": 0,
   "decision": "inconclusive"
  },
  "rule3_beats_frozen_exp1": {
   "a": "setseq_cbd_wide",
   "b": "setseq_bd",
   "mean_delta": 0.001277325631944318,
   "ci95": [
    -0.0009297727073634647,
    0.003819886699793797
   ],
   "wins": 3,
   "decision": "inconclusive"
  }
 },
 "passed": {
  "rule1_centring_helps": false,
  "rule2_grid_edge_mattered": false,
  "rule3_beats_frozen_exp1": false
 },
 "adopt_followup_candidate": false,
 "count_gate": {
  "setseq_cbd_wide": {
   "gain_at_least_005": true,
   "ci_lower_above_zero": true,
   "macro_regression_at_most_001": true,
   "all